In [2]:
# =============================================================================
# EXTERNAL VALIDATION — SHIELD FRAMEWORK ON MODMA DATASET
# =============================================================================
# Private dataset  : E:/cleaned_csv_reduced_430321.csv (91 subjects)
# External dataset : E:/Research/final_eeg_data_MODMA_updated.csv (53 subjects)
# Label column     : MDD (1 = MDD, 0 = HC) — same in both files
# Classifier       : SVM Linear — trained on full private dataset
# Validation       : Direct transfer — no retraining on MODMA
# Seeds            : 42, 0, 1, 7, 99
# Features         : 314 (same pipeline applied to both datasets)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay
)

# ── Output folder ─────────────────────────────────────────────────────────────
OUTPUT_DIR = "external_validation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Settings ──────────────────────────────────────────────────────────────────
SEEDS  = [42, 0, 1, 7, 99]
K      = 30        # consensus k from nested CV
T_CRIT = 2.776     # t-critical for 95% CI, df=4, n=5

# ── MODMA electrode mapping ───────────────────────────────────────────────────
# Standard 10-20 name → MODMA EGI channel number
MODMA_ELECTRODE_MAP = {
    'Fp2': 8,
    'Fz':  10,
    'Fp1': 21,
    'F3':  23,
    'F7':  32,
    'C3':  35,
    'T3':  44,
    'P3':  51,
    'T5':  57,
    'Pz':  61,
    'O1':  69,
    'O2':  82,
    'P4':  91,
    'T6':  95,
    'C4':  103,
    'T4':  107,
    'F8':  121,
    'F4':  123,
    'Cz':  128,
}

# ── Private dataset electrode order (index 0-18) ──────────────────────────────
PRIVATE_ELECTRODE_ORDER = [
    'Fp1','F3','C3','P3','O1','F7','T3','T5',
    'Fp2','F4','C4','P4','O2','F8','T4','T6',
    'Fz','Cz','Pz'
]

# ── Feature types (16 per electrode) ─────────────────────────────────────────
FEATURE_TYPE_LIST = [
    'Alpha_Band_psd_mean',    'Beta_Band_psd_mean',
    'Gamma_Band_psd_mean',    'Theta_Band_psd_mean',
    'Delta_Band_psd_mean',    'Alpha_Band_psd_variance',
    'Beta_Band_psd_variance', 'Gamma_Band_psd_variance',
    'Theta_Band_psd_variance','Delta_Band_psd_variance',
    'Fractal_Dimension',      'Wavelet_Energy',
    'Spectral_Entropy',       'Hjorth_Activity',
    'Hjorth_Mobility',        'Hjorth_Complexity'
]

# ── Extra features (asymmetry + ratio — same names in both datasets) ──────────
EXTRA_FEATURES = [
    'Alpha_Asymmetry_Fp2_minus_Fp1',
    'Alpha_Asymmetry_F4_minus_F3',
    'Alpha_Asymmetry_F8_minus_F7',
    'Theta_Alpha_Ratio_Fp1',
    'Theta_Alpha_Ratio_F3',
    'Theta_Alpha_Ratio_F4',
    'Theta_Alpha_Ratio_Fp2',
    'Theta_Alpha_Ratio_F7',
    'Theta_Alpha_Ratio_F8',
    'Theta_Alpha_Ratio_Fz'
]

# =============================================================================
# STEP 1 — LOAD PRIVATE DATASET
# =============================================================================
print("=" * 70)
print("STEP 1 — Loading private dataset")
print("=" * 70)

PRIVATE_CSV_PATH = r"E:/cleaned_csv_reduced_430321.csv"

private_df = pd.read_csv(PRIVATE_CSV_PATH)

# Drop ID column if present
if 'ID' in private_df.columns:
    private_df = private_df.drop(columns=['ID'])

Y_private = private_df['MDD'].values
X_private_df = private_df.drop(columns=['MDD'])
private_feature_names = list(X_private_df.columns)

print(f"Private dataset shape : {X_private_df.shape}")
print(f"MDD (label=1)         : {np.sum(Y_private == 1)}")
print(f"HC  (label=0)         : {np.sum(Y_private == 0)}")
print(f"Total features        : {len(private_feature_names)}")

# =============================================================================
# STEP 2 — LOAD MODMA DATASET
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2 — Loading MODMA dataset")
print("=" * 70)

MODMA_CSV_PATH = r"E:\Research\final_eeg_data_MODMA_updated.csv"

modma_df = pd.read_csv(MODMA_CSV_PATH)

# Drop ID column
if 'ID' in modma_df.columns:
    modma_df = modma_df.drop(columns=['ID'])

Y_modma    = modma_df['MDD'].values
X_modma_df = modma_df.drop(columns=['MDD'])

print(f"MODMA dataset shape   : {X_modma_df.shape}")
print(f"MDD (label=1)         : {np.sum(Y_modma == 1)}")
print(f"HC  (label=0)         : {np.sum(Y_modma == 0)}")

# =============================================================================
# STEP 3 — BUILD ALIGNED FEATURE MATRICES
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3 — Aligning features to same order")
print("=" * 70)

# ── Build private column names in standard order ──────────────────────────────
# Private uses electrode index 0-18 as suffix: e.g. Alpha_Band_psd_mean_0
private_electrode_cols = []
for elec_idx, electrode in enumerate(PRIVATE_ELECTRODE_ORDER):
    for feat_type in FEATURE_TYPE_LIST:
        col = f"{feat_type}_{elec_idx}"
        private_electrode_cols.append(col)

# ── Build MODMA column names in same order using EGI channel numbers ──────────
modma_electrode_cols = []
for electrode in PRIVATE_ELECTRODE_ORDER:
    egi_num = MODMA_ELECTRODE_MAP[electrode]
    for feat_type in FEATURE_TYPE_LIST:
        col = f"{feat_type}_{egi_num}"
        modma_electrode_cols.append(col)

# ── Verify all columns exist ──────────────────────────────────────────────────
missing_private = [c for c in private_electrode_cols if c not in private_feature_names]
missing_modma   = [c for c in modma_electrode_cols   if c not in X_modma_df.columns]
missing_extra_private = [c for c in EXTRA_FEATURES if c not in private_feature_names]
missing_extra_modma   = [c for c in EXTRA_FEATURES if c not in X_modma_df.columns]

print(f"Private electrode cols missing : {len(missing_private)}")
print(f"MODMA electrode cols missing   : {len(missing_modma)}")
print(f"Private extra cols missing     : {len(missing_extra_private)}")
print(f"MODMA extra cols missing       : {len(missing_extra_modma)}")

if missing_private:
    print("Missing private cols:", missing_private[:5])
if missing_modma:
    print("Missing MODMA cols:", missing_modma[:5])

# ── Extract and stack feature matrices ───────────────────────────────────────
X_private_elec = X_private_df[private_electrode_cols].values   # 304 cols
X_modma_elec   = X_modma_df[modma_electrode_cols].values       # 304 cols

X_private_extra = X_private_df[EXTRA_FEATURES].values          # 10 cols
X_modma_extra   = X_modma_df[EXTRA_FEATURES].values            # 10 cols

X_private_final = np.hstack([X_private_elec, X_private_extra]) # 314 cols
X_modma_final   = np.hstack([X_modma_elec,   X_modma_extra])   # 314 cols

print(f"\nFinal private feature matrix  : {X_private_final.shape}")
print(f"Final MODMA feature matrix    : {X_modma_final.shape}")
assert X_private_final.shape[1] == X_modma_final.shape[1] == 314, \
    "Feature count mismatch — check pipeline"
print("Feature alignment verified: 314 == 314 ✓")

# =============================================================================
# STEP 4 — EXTERNAL VALIDATION ACROSS 5 SEEDS
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4 — External validation across 5 seeds")
print("=" * 70)

results = []

for seed in SEEDS:
    print(f"\n  Seed = {seed}")

    # Build and train pipeline on full private dataset
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("sel",    SelectKBest(score_func=f_classif, k=K)),
        ("clf",    CalibratedClassifierCV(
                       LinearSVC(C=1, class_weight="balanced",
                                 max_iter=10000, random_state=seed),
                       cv=10
                   ))
    ])
    pipe.fit(X_private_final, Y_private)

    # Predict on MODMA — no retraining
    Y_pred  = pipe.predict(X_modma_final)
    Y_proba = pipe.predict_proba(X_modma_final)[:, 1]

    # Compute metrics
    acc  = accuracy_score(Y_modma, Y_pred) * 100
    prec = precision_score(Y_modma, Y_pred, zero_division=0) * 100
    sens = recall_score(Y_modma, Y_pred, zero_division=0) * 100
    f1   = f1_score(Y_modma, Y_pred, zero_division=0) * 100
    auc  = roc_auc_score(Y_modma, Y_proba) * 100

    tn, fp, fn, tp = confusion_matrix(Y_modma, Y_pred).ravel()
    spec = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0.0

    results.append({
        'Seed':        seed,
        'Accuracy':    acc,
        'Precision':   prec,
        'Sensitivity': sens,
        'Specificity': spec,
        'F1':          f1,
        'AUC':         auc,
        'TP': int(tp), 'TN': int(tn),
        'FP': int(fp), 'FN': int(fn)
    })

    print(f"    Acc={acc:.2f}%  Prec={prec:.2f}%  Sens={sens:.2f}%  "
          f"Spec={spec:.2f}%  F1={f1:.2f}%  AUC={auc:.2f}%")
    print(f"    TP={tp}  TN={tn}  FP={fp}  FN={fn}")

# =============================================================================
# STEP 5 — MEAN ± SD AND 95% CI
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5 — Summary statistics (mean ± SD, 95% CI)")
print("=" * 70)

results_df = pd.DataFrame(results)
metrics    = ['Accuracy','Precision','Sensitivity','Specificity','F1','AUC']

summary = {}
for m in metrics:
    vals = results_df[m].values
    mn   = np.mean(vals)
    sd   = np.std(vals, ddof=1)
    se   = sd / np.sqrt(len(vals))
    lo   = mn - T_CRIT * se
    hi   = mn + T_CRIT * se
    summary[m] = {'mean': mn, 'sd': sd, 'ci_lo': lo, 'ci_hi': hi}
    print(f"  {m:<15}: {mn:.2f} ± {sd:.2f}%  "
          f"[95% CI: {lo:.2f}–{hi:.2f}]")

# =============================================================================
# STEP 6 — PAPER-READY TABLE
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6 — Paper-ready table")
print("=" * 70)

print(f"\n{'Metric':<15} {'Mean ± SD':>18} {'95% CI':>22}")
print("-" * 57)
for m in metrics:
    s = summary[m]
    print(f"{m:<15} {s['mean']:>6.2f} ± {s['sd']:>5.2f}%   "
          f"[{s['ci_lo']:.2f}–{s['ci_hi']:.2f}]")

print("\nPer-seed results:")
header = f"{'Seed':<8}" + "".join(f"{m[:5]:>10}" for m in metrics)
print(header)
print("-" * (8 + 10 * len(metrics)))
for _, row in results_df.iterrows():
    line = f"{int(row['Seed']):<8}"
    for m in metrics:
        line += f"{row[m]:>10.2f}"
    print(line)

# =============================================================================
# STEP 7 — SAVE CSV RESULTS
# =============================================================================
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "external_validation_per_seed.csv"),
    index=False
)

summary_rows = []
for m in metrics:
    s = summary[m]
    summary_rows.append({
        'Metric':      m,
        'Mean_%':      round(s['mean'], 2),
        'SD_%':        round(s['sd'],   2),
        'CI_Lo_%':     round(s['ci_lo'], 2),
        'CI_Hi_%':     round(s['ci_hi'], 2),
        'Mean_SD_CI':  f"{s['mean']:.2f} ± {s['sd']:.2f} "
                       f"[{s['ci_lo']:.2f}–{s['ci_hi']:.2f}]"
    })
pd.DataFrame(summary_rows).to_csv(
    os.path.join(OUTPUT_DIR, "external_validation_summary.csv"),
    index=False
)
print(f"\nCSV results saved to {OUTPUT_DIR}/")

# =============================================================================
# STEP 8 — CONFUSION MATRIX (Seed 42)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8 — Confusion matrix figure (Seed 42)")
print("=" * 70)

pipe_42 = Pipeline([
    ("scaler", RobustScaler()),
    ("sel",    SelectKBest(score_func=f_classif, k=K)),
    ("clf",    CalibratedClassifierCV(
                   LinearSVC(C=1, class_weight="balanced",
                             max_iter=10000, random_state=42),
                   cv=5
               ))
])
pipe_42.fit(X_private_final, Y_private)
Y_pred_42 = pipe_42.predict(X_modma_final)

cm = confusion_matrix(Y_modma, Y_pred_42)
fig_cm, ax_cm = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['HC', 'MDD']
)
disp.plot(ax=ax_cm, colorbar=False, cmap='Blues')
ax_cm.set_title(
    "External Validation — MODMA Dataset\n"
    "SHIELD Framework · SVM Linear · Seed = 42",
    fontsize=11, pad=12
)
fig_cm.tight_layout()
for ext in ['png', 'pdf']:
    fig_cm.savefig(
        os.path.join(OUTPUT_DIR, f"confusion_matrix_MODMA_seed42.{ext}"),
        dpi=300, bbox_inches='tight'
    )
plt.close(fig_cm)
print("Confusion matrix saved.")

# =============================================================================
# STEP 9 — COMPARISON BAR CHART: Private Nested CV vs MODMA External
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9 — Comparison bar chart")
print("=" * 70)

# Private nested CV results from your paper
private_nested_cv = {
    'Accuracy':    {'mean': 63.52, 'sd': 2.62},
    'Sensitivity': {'mean': 49.20, 'sd': 3.03},
    'Specificity': {'mean': 80.98, 'sd': 4.69},
    'F1':          {'mean': 59.70, 'sd': 2.85},
    'AUC':         {'mean': 68.76, 'sd': 1.12},
}

plot_metrics = ['Accuracy', 'Sensitivity', 'Specificity', 'F1', 'AUC']
x     = np.arange(len(plot_metrics))
width = 0.35

fig_bar, ax_bar = plt.subplots(figsize=(11, 6))

bars1 = ax_bar.bar(
    x - width/2,
    [private_nested_cv[m]['mean'] for m in plot_metrics],
    width,
    yerr=[private_nested_cv[m]['sd'] for m in plot_metrics],
    label='Private Dataset — Nested CV (n=91)',
    color='#7EB8D4', edgecolor='white',
    capsize=4, alpha=0.9
)
bars2 = ax_bar.bar(
    x + width/2,
    [summary[m]['mean'] for m in plot_metrics],
    width,
    yerr=[summary[m]['sd'] for m in plot_metrics],
    label='MODMA Dataset — External Validation (n=53)',
    color='#85C9A0', edgecolor='white',
    capsize=4, alpha=0.9
)

for bar in list(bars1) + list(bars2):
    ax_bar.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.8,
        f"{bar.get_height():.1f}",
        ha='center', va='bottom', fontsize=8.5
    )

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(plot_metrics, fontsize=11)
ax_bar.set_ylabel("Performance (%)", fontsize=11, labelpad=8)
ax_bar.set_ylim(0, 108)
ax_bar.set_title(
    "SHIELD Framework — Private Dataset vs MODMA External Validation\n"
    "SVM Linear · Mean ± SD across 5 Independent Seeds",
    fontsize=11, pad=14
)
ax_bar.legend(fontsize=9, frameon=True, framealpha=0.9, edgecolor='#ddd')
ax_bar.grid(axis='y', linewidth=0.4, color='#eeeeee')
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
fig_bar.tight_layout()

for ext in ['png', 'pdf']:
    fig_bar.savefig(
        os.path.join(OUTPUT_DIR, f"comparison_private_vs_MODMA.{ext}"),
        dpi=300, bbox_inches='tight'
    )
plt.close(fig_bar)
print("Comparison bar chart saved.")

# =============================================================================
# STEP 10 — FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("EXTERNAL VALIDATION COMPLETE — SHIELD FRAMEWORK")
print("=" * 70)
print(f"\nPrivate dataset  — SVM Linear — Nested CV (n=91):")
print(f"  Accuracy = 63.52 ± 2.62%   AUC = 68.76 ± 1.12%")
print(f"\nMODMA dataset — SVM Linear — External Validation (n=53):")
print(f"  Accuracy = {summary['Accuracy']['mean']:.2f} ± "
      f"{summary['Accuracy']['sd']:.2f}%   "
      f"AUC = {summary['AUC']['mean']:.2f} ± "
      f"{summary['AUC']['sd']:.2f}%")
print(f"\nOutput files saved to: {OUTPUT_DIR}/")
print("  external_validation_per_seed.csv")
print("  external_validation_summary.csv")
print("  confusion_matrix_MODMA_seed42.png/.pdf")
print("  comparison_private_vs_MODMA.png/.pdf")
print("=" * 70)


STEP 1 — Loading private dataset
Private dataset shape : (91, 314)
MDD (label=1)         : 50
HC  (label=0)         : 41
Total features        : 314

STEP 2 — Loading MODMA dataset
MODMA dataset shape   : (53, 314)
MDD (label=1)         : 24
HC  (label=0)         : 29

STEP 3 — Aligning features to same order
Private electrode cols missing : 0
MODMA electrode cols missing   : 0
Private extra cols missing     : 0
MODMA extra cols missing       : 0

Final private feature matrix  : (91, 314)
Final MODMA feature matrix    : (53, 314)
Feature alignment verified: 314 == 314 ✓

STEP 4 — External validation across 5 seeds

  Seed = 42
    Acc=45.28%  Prec=45.28%  Sens=100.00%  Spec=0.00%  F1=62.34%  AUC=53.02%
    TP=24  TN=0  FP=29  FN=0

  Seed = 0
    Acc=45.28%  Prec=45.28%  Sens=100.00%  Spec=0.00%  F1=62.34%  AUC=53.02%
    TP=24  TN=0  FP=29  FN=0

  Seed = 1
    Acc=45.28%  Prec=45.28%  Sens=100.00%  Spec=0.00%  F1=62.34%  AUC=53.02%
    TP=24  TN=0  FP=29  FN=0

  Seed = 7
    Acc=45.

In [12]:
import os
import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# =============================================================================
# CONFIGURATION
# =============================================================================

SEEDS = [42, 0, 1, 7, 99]

TOP_K_FEATURES = 15

PROB_THRESHOLD = 0.60

# =============================================================================
# STEP 1 — LOAD PRIVATE DATASET
# =============================================================================

print("\n" + "="*70)
print("STEP 1 — Loading private dataset")
print("="*70)

private_df = pd.read_csv("E:/cleaned_csv_reduced_430321.csv")

X_private = private_df.drop(columns=["MDD"]).values
y_private = private_df["MDD"].values

print(f"Private dataset shape : {X_private.shape}")
print(f"MDD (label=1)         : {np.sum(y_private == 1)}")
print(f"HC  (label=0)         : {np.sum(y_private == 0)}")

# =============================================================================
# STEP 2 — LOAD MODMA DATASET
# =============================================================================

print("\n" + "="*70)
print("STEP 2 — Loading MODMA dataset")
print("="*70)

modma_df = pd.read_csv("E:/Research/final_eeg_data_MODMA_updated.csv")

X_modma = modma_df.drop(columns=["MDD"]).values
y_modma = modma_df["MDD"].values

print(f"MODMA dataset shape   : {X_modma.shape}")
print(f"MDD (label=1)         : {np.sum(y_modma == 1)}")
print(f"HC  (label=0)         : {np.sum(y_modma == 0)}")

# =============================================================================
# STEP 3 — LOG TRANSFORM
# =============================================================================

print("\n" + "="*70)
print("STEP 3 — Applying log transform")
print("="*70)

X_private = np.log1p(np.abs(X_private))
X_modma = np.log1p(np.abs(X_modma))

print("Log transform applied successfully.")

# =============================================================================
# STEP 4 — EXTERNAL VALIDATION
# =============================================================================

print("\n" + "="*70)
print("STEP 4 — External validation across 5 seeds")
print("="*70)

results = []

for seed in SEEDS:

    print(f"\nSeed = {seed}")

    # =========================================================================
    # FEATURE SELECTION
    # =========================================================================

    selector = SelectKBest(
        score_func=f_classif,
        k=TOP_K_FEATURES
    )

    X_private_selected = selector.fit_transform(X_private, y_private)
    X_modma_selected = selector.transform(X_modma)

    # =========================================================================
    # ROBUST SCALING
    # =========================================================================

    scaler = RobustScaler()

    X_private_scaled = scaler.fit_transform(X_private_selected)
    X_modma_scaled = scaler.transform(X_modma_selected)

    # =========================================================================
    # CLASS-BALANCED LINEAR SVM
    # =========================================================================

    clf = SVC(
        kernel='linear',
        probability=True,
        class_weight='balanced',
        random_state=seed
    )

    clf.fit(X_private_scaled, y_private)

    # =========================================================================
    # PREDICTION
    # =========================================================================

    y_prob = clf.predict_proba(X_modma_scaled)[:, 1]

    y_pred = (y_prob >= PROB_THRESHOLD).astype(int)

    # =========================================================================
    # METRICS
    # =========================================================================

    acc = accuracy_score(y_modma, y_pred) * 100

    prec = precision_score(
        y_modma,
        y_pred,
        zero_division=0
    ) * 100

    sens = recall_score(
        y_modma,
        y_pred
    ) * 100

    tn, fp, fn, tp = confusion_matrix(
        y_modma,
        y_pred
    ).ravel()

    spec = (tn / (tn + fp)) * 100

    f1 = f1_score(
        y_modma,
        y_pred
    ) * 100

    auc = roc_auc_score(
        y_modma,
        y_prob
    ) * 100

    print(f"  Accuracy    = {acc:.2f}%")
    print(f"  Precision   = {prec:.2f}%")
    print(f"  Sensitivity = {sens:.2f}%")
    print(f"  Specificity = {spec:.2f}%")
    print(f"  F1-Score    = {f1:.2f}%")
    print(f"  AUC         = {auc:.2f}%")

    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

    results.append([
        seed,
        acc,
        prec,
        sens,
        spec,
        f1,
        auc,
        tp,
        tn,
        fp,
        fn
    ])

# =============================================================================
# STEP 5 — SUMMARY STATISTICS
# =============================================================================

print("\n" + "="*70)
print("STEP 5 — Summary statistics")
print("="*70)

results_df = pd.DataFrame(results, columns=[
    "Seed",
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC",
    "TP",
    "TN",
    "FP",
    "FN"
])

metrics = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]

for metric in metrics:

    vals = results_df[metric].values

    mean_val = np.mean(vals)
    sd_val = np.std(vals, ddof=1)

    print(f"{metric:15s}: {mean_val:.2f} ± {sd_val:.2f}%")

# =============================================================================
# STEP 6 — SAVE RESULTS
# =============================================================================

os.makedirs("external_validation_results", exist_ok=True)

results_df.to_csv(
    "external_validation_results/external_validation_results.csv",
    index=False
)

print("\nResults saved successfully.")

# =============================================================================
# FINAL MESSAGE
# =============================================================================

print("\n" + "="*70)
print("MODIFIED EXTERNAL VALIDATION COMPLETE")
print("="*70)


STEP 1 — Loading private dataset
Private dataset shape : (91, 315)
MDD (label=1)         : 50
HC  (label=0)         : 41

STEP 2 — Loading MODMA dataset
MODMA dataset shape   : (53, 315)
MDD (label=1)         : 24
HC  (label=0)         : 29

STEP 3 — Applying log transform
Log transform applied successfully.

STEP 4 — External validation across 5 seeds

Seed = 42


ValueError: Input X contains NaN.
SelectKBest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [1]:
# =============================================================================
# CELL 1 — IMPORTS
# =============================================================================
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import time
import pickle
import os
import tempfile

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten,
                                     Dense, Dropout, BatchNormalization, Reshape)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("All imports successful")
print(f"TensorFlow version: {tf.__version__}")

# =============================================================================
# CELL 2 — CONFIGURATION AND DATA LOADING
# =============================================================================
# CHANGE THIS TO YOUR DATA PATH
DATA_PATH     = "E:/Research/final_eeg_data_MODMA_updated.csv"
CONSENSUS_K   = 30
N_OUTER_FOLDS = 10
OUTER_SEEDS   = [42, 0, 1, 7, 99]
N_BOOTSTRAP   = 1000
SEED          = 42

df = pd.read_csv(DATA_PATH)
DROP_COLS = ["MDD", "ID", "Species"]
Y = df["MDD"].values.astype(int)
X = df.drop(columns=DROP_COLS, errors="ignore").values.astype(float)
print(f"Data loaded: X={X.shape}, Y={Y.shape}")
print(f"MDD={np.sum(Y==1)}, HC={np.sum(Y==0)}")

# =============================================================================
# CELL 3 — METRIC FUNCTIONS
# =============================================================================
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy":    accuracy_score(y_true, y_pred),
        "precision":   precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn/(tn+fp) if (tn+fp)>0 else 0.0,
        "f1":          f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":     roc_auc_score(y_true, y_prob)
    }

def bootstrap_ci(y_true, y_pred, y_prob, n=N_BOOTSTRAP):
    mlist = ["accuracy","precision","sensitivity","specificity","f1","roc_auc"]
    res   = {m:[] for m in mlist}
    for i in range(n):
        idx = resample(np.arange(len(y_true)), replace=True,
                       stratify=y_true, random_state=i)
        yt, yp, yb = y_true[idx], y_pred[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        cm = confusion_matrix(yt, yp, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()
        res["accuracy"].append(accuracy_score(yt, yp))
        res["precision"].append(precision_score(yt, yp, zero_division=0))
        res["sensitivity"].append(recall_score(yt, yp, zero_division=0))
        res["specificity"].append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
        res["f1"].append(f1_score(yt, yp, zero_division=0))
        res["roc_auc"].append(roc_auc_score(yt, yb))
    return {m:(np.mean(res[m]),
               np.percentile(res[m], 2.5),
               np.percentile(res[m], 97.5)) for m in mlist}

def get_model_size_kb(obj, is_keras=False):
    """Get model size in KB using pickle for both classical and CNN models"""
    tmp_path = tempfile.mktemp(suffix=".pkl")
    try:
        if is_keras:
            weights = obj.get_weights()
            with open(tmp_path, "wb") as f:
                pickle.dump(weights, f)
        else:
            with open(tmp_path, "wb") as f:
                pickle.dump(obj, f)
        size = os.path.getsize(tmp_path) / 1024
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    return size

print("Metric functions defined")

# =============================================================================
# CELL 4 — CNN MODEL BUILDER
# =============================================================================
def build_cnn(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(32, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(64, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Test CNN build
test_cnn = build_cnn(CONSENSUS_K)
print(f"CNN built successfully")
print(f"CNN parameters: {test_cnn.count_params():,}")
test_cnn.summary()

# =============================================================================
# CELL 5 — CLASSICAL PIPELINE FACTORY
# =============================================================================
def make_pipe(clf_name):
    if clf_name == "SVM":
        clf = SVC(kernel="linear", C=1, class_weight="balanced",
                  probability=True, random_state=SEED)
    elif clf_name == "LR":
        clf = LogisticRegression(C=1, max_iter=3000,
                                  class_weight="balanced", random_state=SEED)
    elif clf_name == "LDA":
        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    elif clf_name == "RF":
        clf = RandomForestClassifier(n_estimators=200, max_depth=4,
                                      class_weight="balanced", random_state=SEED)
    return Pipeline([
        ("scaler", RobustScaler()),
        ("sel",    SelectKBest(f_classif, k=CONSENSUS_K)),
        ("clf",    clf)
    ])

print("Pipeline factory defined")

# =============================================================================
# CELL 6 — MAIN EVALUATION LOOP
# =============================================================================
CLF_NAMES   = ["SVM", "LR", "LDA", "RF", "CNN"]
all_results = {c: {"metrics":[], "ci":[], "train_t":[],
                   "inf_t":[], "size_kb":[]} for c in CLF_NAMES}

for outer_seed in OUTER_SEEDS:
    print(f"\n{'='*50}")
    print(f"OUTER SEED = {outer_seed}")
    print(f"{'='*50}")

    cv    = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True,
                             random_state=outer_seed)
    preds = {c: np.zeros(len(Y), dtype=int) for c in CLF_NAMES}
    probs = {c: np.zeros(len(Y))            for c in CLF_NAMES}
    tt    = {c: [] for c in CLF_NAMES}
    it    = {c: [] for c in CLF_NAMES}
    sz    = {c: [] for c in CLF_NAMES}

    for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(X, Y), 1):
        print(f"  Fold {fold_idx:02d}/{N_OUTER_FOLDS}", end=" | ")
        Xtr, Xte = X[tr_idx], X[te_idx]
        Ytr, Yte = Y[tr_idx], Y[te_idx]

        # --- Classical classifiers ---
        for clf_name in ["SVM", "LR", "LDA", "RF"]:
            pipe = make_pipe(clf_name)

            t0 = time.time()
            pipe.fit(Xtr, Ytr)
            tt[clf_name].append(time.time() - t0)

            t0 = time.time()
            p  = pipe.predict(Xte)
            it[clf_name].append((time.time() - t0) / len(Xte))

            sz[clf_name].append(get_model_size_kb(pipe, is_keras=False))

            probs[clf_name][te_idx] = pipe.predict_proba(Xte)[:, 1]
            preds[clf_name][te_idx] = p

        # --- CNN ---
        sc  = RobustScaler()
        sel = SelectKBest(f_classif, k=CONSENSUS_K)
        Xtr_s = sel.fit_transform(sc.fit_transform(Xtr), Ytr)
        Xte_s = sel.transform(sc.transform(Xte))

        cnn = build_cnn(CONSENSUS_K)
        es  = EarlyStopping(monitor="val_loss", patience=10,
                             restore_best_weights=True, verbose=0)
        cw  = {0: len(Ytr)/(2*np.sum(Ytr==0)),
               1: len(Ytr)/(2*np.sum(Ytr==1))}

        t0 = time.time()
        cnn.fit(Xtr_s, Ytr,
                epochs=100,
                batch_size=16,
                validation_split=0.1,
                callbacks=[es],
                verbose=0,
                class_weight=cw)
        tt["CNN"].append(time.time() - t0)

        t0 = time.time()
        cp = cnn.predict(Xte_s, verbose=0).flatten()
        it["CNN"].append((time.time() - t0) / len(Xte_s))

        sz["CNN"].append(get_model_size_kb(cnn, is_keras=True))

        probs["CNN"][te_idx] = cp
        preds["CNN"][te_idx] = (cp >= 0.5).astype(int)

        print(f"CNN done")

    # Metrics per seed
    for c in CLF_NAMES:
        all_results[c]["metrics"].append(
            compute_metrics(Y, preds[c], probs[c]))
        all_results[c]["ci"].append(
            bootstrap_ci(Y, preds[c], probs[c]))
        all_results[c]["train_t"].append(np.mean(tt[c]))
        all_results[c]["inf_t"].append(np.mean(it[c]))
        all_results[c]["size_kb"].append(np.mean(sz[c]))

    # Per-seed summary
    print(f"\nSeed {outer_seed} summary:")
    for c in CLF_NAMES:
        m = all_results[c]["metrics"][-1]
        print(f"  {c:<6} AUC={m['roc_auc']:.3f} "
              f"Acc={m['accuracy']*100:.2f}% "
              f"Sens={m['sensitivity']*100:.2f}% "
              f"Spec={m['specificity']*100:.2f}%")

print("\nAll seeds complete!")

# =============================================================================
# CELL 7 — PRINT FINAL RESULTS
# =============================================================================
MLIST = ["roc_auc","accuracy","precision","sensitivity","specificity","f1"]

print("\n" + "="*110)
print("CLASSIFICATION PERFORMANCE — Mean +- SD across 5 seeds")
print("AUC: 0-1 scale | All other metrics: percentage (%)")
print("="*110)
print(f"{'Classifier':<10} {'AUC (0-1)':>14} {'Acc%':>14} "
      f"{'Prec%':>14} {'Sens%':>14} {'Spec%':>14} {'F1%':>12}")
print("-"*110)

for c in CLF_NAMES:
    v = {m: [r[m] for r in all_results[c]["metrics"]] for m in MLIST}
    print(
        f"{c:<10} "
        f"{np.mean(v['roc_auc']):>6.3f}+-{np.std(v['roc_auc'],ddof=1):>5.3f}  "
        f"{np.mean(v['accuracy'])*100:>8.2f}+-{np.std(v['accuracy'],ddof=1)*100:>4.2f}  "
        f"{np.mean(v['precision'])*100:>8.2f}+-{np.std(v['precision'],ddof=1)*100:>4.2f}  "
        f"{np.mean(v['sensitivity'])*100:>8.2f}+-{np.std(v['sensitivity'],ddof=1)*100:>4.2f}  "
        f"{np.mean(v['specificity'])*100:>8.2f}+-{np.std(v['specificity'],ddof=1)*100:>4.2f}  "
        f"{np.mean(v['f1'])*100:>6.2f}+-{np.std(v['f1'],ddof=1)*100:>4.2f}"
    )

# =============================================================================
# CELL 8 — 95% CONFIDENCE INTERVALS
# =============================================================================
print("\n" + "="*100)
print("95% BOOTSTRAP CONFIDENCE INTERVALS — Mean across 5 seeds")
print("AUC: 0-1 scale | Other metrics: percentage (%)")
print("="*100)

for c in CLF_NAMES:
    print(f"\n{c}:")
    for m in MLIST:
        means = [r[m][0] for r in all_results[c]["ci"]]
        lows  = [r[m][1] for r in all_results[c]["ci"]]
        highs = [r[m][2] for r in all_results[c]["ci"]]
        if m == "roc_auc":
            print(f"  AUC:           {np.mean(means):.3f} "
                  f"[{np.mean(lows):.3f} - {np.mean(highs):.3f}]")
        else:
            print(f"  {m:<14}  {np.mean(means)*100:6.2f}% "
                  f"[{np.mean(lows)*100:6.2f}% - {np.mean(highs)*100:6.2f}%]")

# =============================================================================
# CELL 9 — COMPUTATIONAL COMPLEXITY
# =============================================================================
print("\n" + "="*80)
print("COMPUTATIONAL COMPLEXITY — Mean across 5 seeds x 10 folds")
print("="*80)
print(f"{'Classifier':<12} {'Train Time(s)':>16} "
      f"{'Inference(ms/subj)':>22} {'Model Size(KB)':>16}")
print("-"*80)

for c in CLF_NAMES:
    print(
        f"{c:<12} "
        f"{np.mean(all_results[c]['train_t']):>14.4f}s  "
        f"{np.mean(all_results[c]['inf_t'])*1000:>18.4f}ms  "
        f"{np.mean(all_results[c]['size_kb']):>14.2f}KB"
    )

print("="*80)
print("\nNotes:")
print("  CNN = 1D Convolutional Neural Network on k=30 SHIELD selected features")
print("  All classifiers evaluated under identical leakage-free nested CV")
print("  Train time = mean per fold | Inference = per subject | Size = weights")

All imports successful
TensorFlow version: 2.10.0
Data loaded: X=(53, 314), Y=(53,)
MDD=24, HC=29
Metric functions defined
CNN built successfully
CNN parameters: 35,521
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 30, 1)             0         
                                                                 
 conv1d (Conv1D)             (None, 30, 32)            128       
                                                                 
 batch_normalization (BatchN  (None, 30, 32)           128       
 ormalization)                                                   
                                                                 
 max_pooling1d (MaxPooling1D  (None, 15, 32)           0         
 )                                                               
                                                                 
 conv1d_1 (Conv1D) 